<a href="https://colab.research.google.com/github/pdnivedh/pathoGen/blob/main/notebooks/01_biobert_training_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
!git clone https://github.com/pdnivedh/pathoGen.git
%cd pathoGen

Cloning into 'pathoGen'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 40 (delta 16), reused 30 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 84.89 KiB | 7.72 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/pathoGen/pathoGen/pathoGen


In [3]:
!pip install -q transformers datasets scikit-learn accelerate

In [6]:
!python src/train_biobert.py "/content/drive/MyDrive/pathogen data" /content/drive/MyDrive/pathogen/biobert_v2

config.json: 100% 1.11k/1.11k [00:00<00:00, 3.62MB/s]
vocab.txt: 100% 213k/213k [00:00<00:00, 37.8MB/s]
Map: 100% 807/807 [00:00<00:00, 3030.05 examples/s]
Map: 100% 173/173 [00:00<00:00, 3019.57 examples/s]
[transformers] You passed `num_labels=24` which is incompatible to the `id2label` map of length `2`.

pytorch_model.bin: downloading bytes:  92% 401M/436M [00:02<00:00, 168MB/s, 35.1MB/s  ]
pytorch_model.bin: downloading bytes: 100% 413M/413M [00:04<00:00, 84.8MB/s, 35.6MB/s  ]
pytorch_model.bin: reconstructing file: 100% 436M/436M [00:04<00:00, 89.5MB/s, 38.8MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 24624.34it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerN

In [10]:
!python src/uncertainty.py "/content/drive/MyDrive/pathogen/biobert_v2/best" "/content/drive/MyDrive/pathogen data/val.csv" "/content/drive/MyDrive/pathogen/val_uncertainty.csv"

Loading weights: 100% 201/201 [00:00<00:00, 2597.60it/s]
Deterministic: acc=0.9942 f1=0.9948 ECE=0.0318 mean_conf=0.9650
MC Dropout: acc=0.9942 f1=0.9948 ECE=0.0448 mean_conf=0.9518

Misclassified (MC):
       true pred_mc   conf_mc  entropy_mc
13  typhoid  dengue  0.758925    0.971125

10 least confident (MC):
                 true           pred_mc   conf_mc  entropy_mc
1       drug reaction     drug reaction  0.441924    1.610777
115         psoriasis         psoriasis  0.462307    1.463523
165       chicken pox       chicken pox  0.517951    0.934170
64          psoriasis         psoriasis  0.574157    1.452328
130            dengue            dengue  0.591886    1.534109
156  bronchial asthma  bronchial asthma  0.704501    1.332540
162           allergy           allergy  0.745391    1.053806
13            typhoid            dengue  0.758925    0.971125
85           diabetes          diabetes  0.804984    0.937253
24           diabetes          diabetes  0.848293    0.727657


In [12]:
!python src/threshold_analysis.py "/content/drive/MyDrive/pathogen/val_uncertainty.csv"

n=173, errors=1

thr   flagged  flag%  acc_on_kept  errors_caught
0.50       2    1.2       0.9942  0/1
0.60       5    2.9       0.9940  0/1
0.70       5    2.9       0.9940  0/1
0.80       8    4.6       1.0000  1/1
0.85      10    5.8       1.0000  1/1
0.90      16    9.2       1.0000  1/1
0.95      23   13.3       1.0000  1/1


In [13]:
!python src/uncertainty.py "/content/drive/MyDrive/pathogen/biobert_v2/best" "/content/drive/MyDrive/pathogen data/test.csv" "/content/drive/MyDrive/pathogen/test_uncertainty.csv"

Loading weights: 100% 201/201 [00:00<00:00, 2631.87it/s]
Deterministic: acc=0.9827 f1=0.9828 ECE=0.0310 mean_conf=0.9674
MC Dropout: acc=0.9827 f1=0.9828 ECE=0.0419 mean_conf=0.9546

Misclassified (MC):
            true        pred_mc   conf_mc  entropy_mc
25      diabetes  drug reaction  0.824534    0.657944
64   chicken pox         dengue  0.374276    1.728143
160      typhoid    chicken pox  0.420839    1.519734

10 least confident (MC):
                 true           pred_mc   conf_mc  entropy_mc
64        chicken pox            dengue  0.374276    1.728143
160           typhoid       chicken pox  0.420839    1.519734
123          impetigo          impetigo  0.601406    1.240599
120  bronchial asthma  bronchial asthma  0.702716    1.175300
151       chicken pox       chicken pox  0.721724    0.818436
17      drug reaction     drug reaction  0.724186    0.973560
29           impetigo          impetigo  0.760069    0.944437
107         arthritis         arthritis  0.788225    0.9569

In [14]:
!python src/threshold_analysis.py "/content/drive/MyDrive/pathogen/test_uncertainty.csv"

n=173, errors=3

thr   flagged  flag%  acc_on_kept  errors_caught
0.50       2    1.2       0.9942  2/3
0.60       2    1.2       0.9942  2/3
0.70       3    1.7       0.9941  2/3
0.80       9    5.2       0.9939  2/3
0.85      11    6.4       1.0000  3/3
0.90      13    7.5       1.0000  3/3
0.95      23   13.3       1.0000  3/3
